In [ ]:
# Rossenblat Perceptron with No Activation layer or any of the improvements, the OG one:

# Shity psuedo code
# for epoch in 1..max_epochs:
#     err = 0
#     for each sample (xi, yi):
#         z  = w · xi + b
#         ŷi = 1 if z ≥ 0 else 0
#         if ŷi ≠ yi: err += 1
#         wj = wj + lr · (yi − ŷi) · xij
#         b  = b  + lr · (yi − ŷi)
#     if err == 0: stop

# THis is one of the flaws of percptron, if we cant linearly seperate a data, the epoch will run forever cuz we depend on this err thing to be 0, while models like Adaline instead depend on the MSE and Gradient Descent.

In [2]:
from tinygrad import Tensor

In [27]:


def train(X: Tensor, Y: Tensor, max_epochs: 100, lr: 2.0):

   w,b = Tensor.randint(X.shape[1]), Tensor(0.)

   history = []

   for _ in range(max_epochs):
        
      errors = Tensor(0)

      for x, y in zip(X, Y):

         # The step function 
         pred_y = ((x @ w) + b >= 0).where(1., 0.)

         err = y - pred_y

         # We force update
         w, b, errors = w + lr * err * x, b + lr * err, errors + (err != 0)

         Tensor.realize(w,b, errors)

      history.append(errors.item())

      if history[-1] == 0: break

   return w,b, history

In [ ]:
X = Tensor([[0.,0.], [0.,1.], [1.,0.], [1.,1.]])
targets = {"AND": [0.,0.,0.,1.], "OR": [0.,1.,1.,1.], "XNOR": [1.,0.,0.,1.]}

# LLM generated
for name, labels in targets.items():
    T = Tensor(labels)                              # flat (4,) targets
    weights, bias, history = train(X, T, max_epochs=20, lr=2.0)
    preds = (X @ weights + bias >= 0).where(1., 0.) # all 4 rows in one step
    status = "converged" if history[-1] == 0 else "hit max_epochs"
    print(f"{name}: {status} after {len(history)} epochs | errors {history} | preds {preds.tolist()} | targets {labels}")

AND: converged after 5 epochs | errors [3, 2, 2, 1, 0] | preds [0.0, 0.0, 0.0, 1.0] | targets [0.0, 0.0, 0.0, 1.0]
OR: converged after 2 epochs | errors [1, 0] | preds [0.0, 1.0, 1.0, 1.0] | targets [0.0, 1.0, 1.0, 1.0]
XNOR: hit max_epochs after 20 epochs | errors [2, 3, 3, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4] | preds [0.0, 0.0, 1.0, 1.0] | targets [1.0, 0.0, 0.0, 1.0]
